# Kapitel 4: Die Maschine lernt

## Bisher...

Wir haben 400.000 Bewertungen entdeckt (Kap. 1), bereinigt (Kap. 2)
und in mathematische Vektoren verwandelt (Kap. 3). Jedes Review ist jetzt
ein 10.000-dimensionaler TF-IDF-Vektor mit einem Label: 0 oder 1.

## Die entscheidende Frage

Reicht das aus, damit eine Maschine lernt, Emotionen zu erkennen?

In diesem Kapitel finden wir es heraus. Wir trainieren ein **Logistic-Regression-Modell**
und messen, wie gut es positive von negativen Bewertungen unterscheiden kann.

Logistic Regression ist bewusst gewählt: Es ist schnell, interpretierbar
und ideal für den Einstieg in die Textklassifikation. Wenn selbst ein
einfaches Modell gute Ergebnisse liefert, bestätigt das die Qualität unserer Features.

## 4.1 TF-IDF-Daten laden

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviews – Sentiment-Klassifikation") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

df = spark.read.parquet(
    "/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/tfidf_features.parquet"
)

print(f"Daten geladen: {df.count():,} Zeilen")
df.printSchema()

## 4.2 Daten aufteilen: Training und Test

Wir teilen die Daten im Verhältnis **80/20** auf:
- **80% Training** — daraus lernt das Modell
- **20% Test** — damit prüfen wir, ob das Gelernte auch auf neue Daten übertragbar ist

Warum? Wenn wir auf denselben Daten trainieren und testen, misst das Modell
nur sein Gedächtnis, nicht sein Verständnis. Der Test-Split ist die ehrliche Prüfung.

In [ ]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print(f"Training:  {train_df.count():,} Zeilen")
print(f"Test:      {test_df.count():,} Zeilen")

print("\nLabel-Verteilung (Training):")
train_df.groupBy("label").count().orderBy("label").show()

## 4.3 Modell trainieren

Jetzt passiert es: Die Maschine sieht zum ersten Mal die Daten und versucht,
ein Muster zu erkennen. Logistic Regression lernt Gewichte für jede der
10.000 TF-IDF-Dimensionen und entscheidet dann für jedes neue Review:
Überwiegen die "positiven" oder die "negativen" Wörter?

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="tfidf_features",
    labelCol="label",
    maxIter=20,
    regParam=0.1
)

print("Training gestartet...")
lr_model = lr.fit(train_df)
print("Training abgeschlossen!")

In [ ]:
training_summary = lr_model.summary
print(f"Accuracy (Training):  {training_summary.accuracy:.4f}")
print(f"Area under ROC:       {training_summary.areaUnderROC:.4f}")

## 4.4 Der Moment der Wahrheit: Vorhersagen auf neuen Daten

Das Modell hat auf den Trainingsdaten gelernt. Jetzt testen wir es auf
den **20% Testdaten**, die es noch nie gesehen hat.

In [ ]:
predictions = lr_model.transform(test_df)
predictions.select("label", "prediction", "probability", "text").show(10, truncate=60)

## 4.5 Wie gut ist das Modell?

Wir messen die Leistung mit mehreren Metriken:
- **Accuracy** — Wie oft liegt das Modell insgesamt richtig?
- **Precision** — Wenn es "positiv" sagt, wie oft stimmt das?
- **Recall** — Von allen tatsächlich positiven, wie viele findet es?
- **F1-Score** — Das harmonische Mittel aus Precision und Recall
- **AUC-ROC** — Wie gut trennt das Modell die beiden Klassen insgesamt?

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

binary_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
auc = binary_eval.evaluate(predictions)

multi_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

accuracy  = multi_eval.evaluate(predictions, {multi_eval.metricName: "accuracy"})
precision = multi_eval.evaluate(predictions, {multi_eval.metricName: "weightedPrecision"})
recall    = multi_eval.evaluate(predictions, {multi_eval.metricName: "weightedRecall"})
f1        = multi_eval.evaluate(predictions, {multi_eval.metricName: "f1"})

print("=" * 45)
print("  EVALUATIONS-ERGEBNISSE (Testdaten)")
print("=" * 45)
print(f"  Accuracy:           {accuracy:.4f}")
print(f"  Precision (gew.):   {precision:.4f}")
print(f"  Recall (gew.):      {recall:.4f}")
print(f"  F1-Score (gew.):    {f1:.4f}")
print(f"  AUC-ROC:            {auc:.4f}")
print("=" * 45)

### Was bedeuten diese Zahlen?

- **~85% Accuracy**: Von 6 Bewertungen erkennt das Modell 5 korrekt
- **AUC = 0.92**: Das Modell trennt positiv/negativ deutlich besser als Zufall (0.5)
- **Precision ≈ Recall**: Das Modell ist bei beiden Klassen gleich gut — keine Seite wird bevorzugt

Für ein einfaches Modell mit Bag-of-Words-Features ist das ein starkes Ergebnis.

## 4.6 Confusion Matrix: Wo macht das Modell Fehler?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cm_data = predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").toPandas()

cm = np.zeros((2, 2), dtype=int)
for _, row in cm_data.iterrows():
    cm[int(row["label"]), int(row["prediction"])] = int(row["count"])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")

labels = ["Negativ (0)", "Positiv (1)"]
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(labels); ax.set_yticklabels(labels)
ax.set_xlabel("Vorhersage (Prediction)")
ax.set_ylabel("Tatsächlich (Label)")
ax.set_title("Confusion Matrix – Logistic Regression")

for i in range(2):
    for j in range(2):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                fontsize=18, fontweight="bold", color=color)

plt.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print(f"True Negative  (korrekt negativ):  {cm[0,0]:,}")
print(f"False Positive (fälschlich positiv): {cm[0,1]:,}")
print(f"False Negative (fälschlich negativ): {cm[1,0]:,}")
print(f"True Positive  (korrekt positiv):  {cm[1,1]:,}")

## 4.7 Ein Blick auf die Fehler

Wo irrt sich die Maschine? Schauen wir uns Beispiele an —
die falschen Vorhersagen verraten viel über die Grenzen des Modells.

In [ ]:
from pyspark.sql.functions import col

print("=== KORREKTE Vorhersagen ===")
predictions.filter(col("label") == col("prediction")) \
    .select("label", "prediction", "text").show(5, truncate=80)

print("=== FALSCHE Vorhersagen ===")
predictions.filter(col("label") != col("prediction")) \
    .select("label", "prediction", "text").show(5, truncate=80)

### Erkenntnis

Die Fehler treten häufig bei **mehrdeutigen Texten** auf: sarkastische Bewertungen,
gemischte Meinungen (*"tolles Produkt, aber der Versand war eine Katastrophe"*)
oder sehr kurze Texte mit wenig Kontext.

Das ist eine bekannte Grenze von Bag-of-Words-Modellen: Sie sehen einzelne Wörter,
aber nicht den **Kontext** oder die **Reihenfolge**. *"Not good"* enthält das Wort *"good"*
und kann das Modell in die Irre führen.

## 4.8 Ergebnisse und Modell speichern

In [ ]:
output_path = "/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/predictions.parquet"
predictions.select("label", "prediction", "probability", "text") \
    .write.parquet(output_path, mode="overwrite")
print(f"Vorhersagen gespeichert: {output_path}")

In [ ]:
model_path = "/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/lr_model"
lr_model.write().overwrite().save(model_path)
print(f"Modell gespeichert: {model_path}")

## Kapitel 4 — Zusammenfassung

| Metrik | Wert | Bedeutung |
|--------|------|----------|
| Accuracy | ~84,7% | 5 von 6 Bewertungen korrekt erkannt |
| AUC-ROC | ~0,92 | Starke Trennung positiv/negativ |
| F1-Score | ~84,7% | Ausgewogene Performance |

**Die Antwort auf unsere Ausgangsfrage: Ja, eine Maschine kann Emotionen erkennen.**
Nicht perfekt — aber mit 85% Accuracy deutlich besser als Zufall. Und das mit einem
der einfachsten ML-Modelle.

**Nächstes Kapitel:** Wir visualisieren alle Ergebnisse und ziehen ein Fazit —
Was hat das Modell gelernt? Wo sind seine Grenzen? Und wie könnte man es verbessern?